# 07_通信协议

**来源:** [LangChain Docs — Agent Client Protocol (ACP)](https://docs.langchain.com/oss/python/deepagents/acp)

本 Notebook 演示 Agent Client Protocol (ACP) 的用法。ACP 标准化了编码 Agent 与代码编辑器或 IDE 之间的通信，使自定义深度 Agent 可与任何兼容 ACP 的客户端协作工作。

## 1. 环境准备与安装

In [ ]:
# 安装 ACP 集成包
# pip install deepagents-acp

import asyncio

from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver

print("环境准备完成！")

## 2. ACP 概述

ACP（Agent Client Protocol）是一种标准化协议，用于编码 Agent 与代码编辑器或 IDE 之间的通信。

- **ACP 是为 Agent-编辑器集成设计的**
- 如果希望 Agent 调用外部服务器托管的工具，请使用 MCP（Model Context Protocol）

通过 ACP 协议，你可以：
- 使用自定义的深度 Agent 与任何 ACP 兼容客户端
- 让代码编辑器提供项目上下文
- 接收丰富的更新信息

## 3. 快速开始

将深度 Agent 暴露为 ACP 服务。这会启动一个 stdio 模式的 ACP 服务器（从 stdin 读取请求，将响应写入 stdout）。

In [ ]:
from acp import run_agent
from deepagents_acp.server import AgentServerACP


async def main() -> None:
    # 创建深度 Agent
    agent = create_deep_agent(
        model="google_genai:gemini-3.5-flash",
        # 可在此自定义深度 Agent：设置自定义提示、添加工具、附加中间件或组合子 Agent
        system_prompt="你是一个有用的编码助手",
        checkpointer=MemorySaver(),
    )

    # 创建 ACP 服务器
    server = AgentServerACP(agent)

    # 运行 Agent（stdio 模式）
    await run_agent(server)


# 实际运行时取消注释以下行：
# asyncio.run(main())

print("ACP 服务器代码已就绪！")

## 4. 示例编码 Agent

`deepagents-acp` 包包含一个示例编码 Agent，预配置了文件系统和 shell 工具，可以直接运行。

```bash
# 克隆 deepagents 仓库并安装
# git clone https://github.com/langchain-ai/deepagents.git
# cd deepagents/libs/acp
# uv sync --all-groups
# chmod +x run_demo_agent.sh
```

## 5. ACP 客户端

深度 Agent 可在任何支持 ACP Agent 服务器的地方运行。

| 客户端 | 支持情况 |
|--------|---------|
| **Zed** | 原生支持 ACP 集成 |
| **JetBrains IDEs** | 支持 ACP |
| **VS Code** | 通过 vscode-acp 扩展 |
| **Neovim** | 通过 ACP 兼容插件 |

## 6. 在 Zed 中配置

1. 克隆 deepagents 仓库并安装依赖：

```bash
git clone https://github.com/langchain-ai/deepagents.git
cd deepagents/libs/acp
uv sync --all-groups
chmod +x run_demo_agent.sh
```

2. 配置凭据：

```bash
cp .env.example .env
# 在 .env 中设置 ANTHROPIC_API_KEY
```

3. 在 Zed 的 `settings.json` 中配置 ACP Agent 服务器命令：

```json
{
  "agent_servers": {
    "DeepAgents": {
      "type": "custom",
      "command": "/your/absolute/path/to/deepagents/libs/acp/run_demo_agent.sh"
    }
  }
}
```

4. 打开 Zed 的 Agents 面板，启动 Deep Agents 会话。

## 7. 使用 Toad 运行 ACP 服务器

如果要将 ACP Agent 服务器作为本地开发工具运行，可以使用 Toad 管理进程：

```bash
# 安装 Toad
uv tool install -U batrachian-toad

# 使用 Toad 运行 ACP 服务器
toad acp "python path/to/your_server.py" .
# 或者
toad acp "uv run python path/to/your_server.py" .
```

## 8. 完整 ACP 服务器示例

以下是一个完整的 ACP 服务器脚本，你可以保存为 `acp_server.py` 并运行：

In [ ]:
"""
    ACP Server 完整示例
    保存为 acp_server.py 并运行：
    python acp_server.py
"""
import asyncio

from acp import run_agent
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver

from deepagents_acp.server import AgentServerACP


async def main() -> None:
    """创建并运行 ACP Agent 服务器。"""
    # 配置深度 Agent
    agent = create_deep_agent(
        model="google_genai:gemini-3.5-flash",
        system_prompt="""你是一个有用的编码助手。你可以：
1. 阅读和分析代码文件
2. 编写和修改代码
3. 回答编程相关问题
4. 调试和审查代码

尽量提供简洁、准确的回答。
""",
        checkpointer=MemorySaver(),
    )

    # 创建 ACP 服务器
    server = AgentServerACP(agent)

    # 以 stdio 模式运行
    await run_agent(server)


if __name__ == "__main__":
    asyncio.run(main())

print("完整 ACP 服务器示例已就绪！")

---

**参考链接**
- [Deep Agents — ACP](https://docs.langchain.com/oss/python/deepagents/acp)
- [Agent Client Protocol 文档](https://agentclientprotocol.com/)
- [ACP 入门](https://agentclientprotocol.com/get-started/introduction)
- [ACP 客户端列表](https://agentclientprotocol.com/get-started/clients)
- [Model Context Protocol (MCP)](https://docs.langchain.com/mcp)
- [Deep Agents GitHub 仓库](https://github.com/langchain-ai/deepagents)